In [ ]:
# notebooks/02_complete_models.ipynb
"""
Complete Task 2: ARIMA and LSTM Model Comparison
With full error handling and comparison table
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

# Import LSTM
import sys
sys.path.append('..')
from src.lstm_model import LSTMModel

print("="*60)
print("TASK 2: ARIMA VS LSTM COMPARISON")
print("="*60)

# Load TSLA data
def load_tsla_data():
    try:
        df = pd.read_csv('../data/processed/TSLA_data.csv', parse_dates=['Date'])
        df.set_index('Date', inplace=True)
        return df['Close']
    except Exception as e:
        print(f"Error loading data: {e}")
        raise

series = load_tsla_data()
print(f"\nLoaded {len(series)} rows of TSLA data")

# Train/test split (chronological)
train_size = int(len(series) * 0.8)
train, test = series[:train_size], series[train_size:]

print(f"\nData Split:")
print(f"  Training: {len(train)} rows ({train.index.min()} to {train.index.max()})")
print(f"  Test: {len(test)} rows ({test.index.min()} to {test.index.max()})")

# 1. ARIMA Model
def fit_arima(train_data, test_data, order=(2,1,2)):
    """Fit ARIMA with error handling."""
    try:
        model = ARIMA(train_data, order=order)
        fitted = model.fit()
        predictions = fitted.forecast(len(test_data))
        
        # Calculate metrics
        mae = mean_absolute_error(test_data, predictions)
        rmse = np.sqrt(mean_squared_error(test_data, predictions))
        mape = np.mean(np.abs((test_data - predictions) / test_data)) * 100
        
        return {
            'model': fitted,
            'predictions': predictions,
            'MAE': mae,
            'RMSE': rmse,
            'MAPE': mape,
            'order': order
        }
    except Exception as e:
        print(f"ARIMA failed: {e}")
        return None

print("\n" + "="*60)
print("ARIMA MODEL")
print("="*60)

arima_results = fit_arima(train, test, order=(2,1,2))
if arima_results:
    print(f"\nARIMA(2,1,2) Performance:")
    print(f"  MAE: ${arima_results['MAE']:.2f}")
    print(f"  RMSE: ${arima_results['RMSE']:.2f}")
    print(f"  MAPE: {arima_results['MAPE']:.2f}%")

# 2. LSTM Model
print("\n" + "="*60)
print("LSTM MODEL")
print("="*60)

def train_lstm(train_data, test_data, sequence_length=60):
    """Train and evaluate LSTM with error handling."""
    try:
        lstm = LSTMModel(sequence_length=sequence_length)
        lstm.fit(train_data, epochs=30, batch_size=32)
        evaluation = lstm.evaluate(train_data.iloc[-sequence_length:], test_data)
        return lstm, evaluation
    except Exception as e:
        print(f"LSTM training failed: {e}")
        return None, None

lstm_model, lstm_results = train_lstm(train, test)

if lstm_results:
    print(f"\nLSTM Performance:")
    print(f"  MAE: ${lstm_results['MAE']:.2f}")
    print(f"  RMSE: ${lstm_results['RMSE']:.2f}")
    print(f"  MAPE: {lstm_results['MAPE']:.2f}%")

# 3. Comparison Table
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

comparison_data = []

if arima_results:
    comparison_data.append({
        'Model': 'ARIMA(2,1,2)',
        'MAE': f"${arima_results['MAE']:.2f}",
        'RMSE': f"${arima_results['RMSE']:.2f}",
        'MAPE': f"{arima_results['MAPE']:.2f}%"
    })

if lstm_results:
    comparison_data.append({
        'Model': 'LSTM',
        'MAE': f"${lstm_results['MAE']:.2f}",
        'RMSE': f"${lstm_results['RMSE']:.2f}",
        'MAPE': f"{lstm_results['MAPE']:.2f}%"
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Best model selection
if arima_results and lstm_results:
    best_model = 'ARIMA' if arima_results['RMSE'] < lstm_results['RMSE'] else 'LSTM'
    print(f"\n✓ Best performing model: {best_model}")
    print("  Selected for forecasting in Task 3")

# 4. Visualizations
fig, ax = plt.subplots(figsize=(14, 7))

# Plot training data
ax.plot(train.index, train, label='Training Data', linewidth=1, alpha=0.7, color='blue')

# Plot test actual
ax.plot(test.index, test, label='Test Actual', linewidth=2, color='green')

# Plot predictions
if arima_results:
    ax.plot(test.index, arima_results['predictions'], 
            label=f"ARIMA(2,1,2)", linewidth=2, linestyle='--', color='red')

if lstm_results:
    lstm_pred_dates = test.index[:len(lstm_results['predictions'])]
    ax.plot(lstm_pred_dates, lstm_results['predictions'][:len(lstm_pred_dates)], 
            label='LSTM', linewidth=2, linestyle=':', color='orange')

ax.set_title('ARIMA vs LSTM: TSLA Price Predictions', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("TASK 2 COMPLETE")
print("="*60)